In [1]:
import os
import sys
import glob
import time
import socket
import shutil
import subprocess
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

YOUTUBE_URL = "https://youtu.be/Qd4jGgu06dw"

WORKDIR = Path("/kaggle/working")
BGUTIL_DIR = WORKDIR / "bgutil-ytdlp-pot-provider"
SERVER_DIR = BGUTIL_DIR / "server"

# Use our own port so an old/background process cannot interfere
BGUTIL_PORT = 4417

OUTPUT_TEMPLATE = str(WORKDIR / "test_audio.%(ext)s")
SERVER_LOG = WORKDIR / "bgutil-server.log"


# ============================================================
# HELPER
# ============================================================

def run(cmd, cwd=None, check=True, env=None):
    print("\n>", " ".join(map(str, cmd)))
    return subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        check=check,
        env=env,
    )


def clean_audio():
    for f in glob.glob(str(WORKDIR / "test_audio.*")):
        try:
            os.remove(f)
        except:
            pass


def find_audio():
    files = glob.glob(str(WORKDIR / "test_audio.*"))
    return files[0] if files else None


# ============================================================
# 0. SAFE WORKING DIRECTORY
# ============================================================

WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

print("=" * 70)
print("KAGGLE YOUTUBE AUDIO — FULL CLEAN BOOTSTRAP")
print("=" * 70)
print("Working directory:", os.getcwd())


# ============================================================
# 1. INSTALL EXACT WORKING PYTHON PACKAGES
# ============================================================

print("\n[1/8] Installing yt-dlp + BgUtils plugin...")

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "yt-dlp[default]==2026.8.19",
    "bgutil-ytdlp-pot-provider==2.0.0",
])


# ============================================================
# 2. INSTALL / LOCATE DENO
# ============================================================

print("\n[2/8] Checking Deno...")

deno = shutil.which("deno")

if not deno:
    print("Deno missing — installing Deno...")

    run([
        "bash",
        "-c",
        "curl -fsSL https://deno.land/install.sh | sh"
    ])

    deno_dir = str(Path.home() / ".deno" / "bin")
    os.environ["PATH"] = deno_dir + ":" + os.environ["PATH"]

    deno = shutil.which("deno")

if not deno:
    raise RuntimeError("Deno installation failed")

print(
    subprocess.check_output(
        [deno, "--version"],
        text=True
    ).splitlines()[0]
)


# ============================================================
# 3. GET MATCHING BGUTIL 2.0.0 SOURCE
# ============================================================

print("\n[3/8] Preparing BgUtils 2.0.0...")

# IMPORTANT:
# we are standing in /kaggle/working before deleting BGUTIL_DIR
os.chdir(WORKDIR)

if BGUTIL_DIR.exists():
    shutil.rmtree(BGUTIL_DIR)

run([
    "git",
    "clone",
    "--depth", "1",
    "--single-branch",
    "--branch", "2.0.0",
    "https://github.com/Brainicism/bgutil-ytdlp-pot-provider.git",
    BGUTIL_DIR,
])

if not SERVER_DIR.exists():
    raise RuntimeError("BgUtils server directory was not cloned correctly")


# ============================================================
# 4. INSTALL BGUTIL'S ACTUAL JS DEPENDENCIES
#    THIS IS WHAT WAS MISSING BEFORE
# ============================================================

print("\n[4/8] Installing BgUtils Deno/npm dependencies...")

deno_env = os.environ.copy()
deno_env["DENO_NO_PROMPT"] = "1"
deno_env["DENO_NO_UPDATE_CHECK"] = "1"

run([
    deno,
    "install",
    "--allow-scripts=npm:canvas",
    "--frozen",
], cwd=SERVER_DIR, env=deno_env)

node_modules = SERVER_DIR / "node_modules"

if not node_modules.exists():
    raise RuntimeError(
        "BgUtils dependency installation finished but node_modules is missing"
    )

print("Dependencies installed correctly.")
print("node_modules:", node_modules)


# ============================================================
# 5. START BGUTIL HTTP TOKEN SERVER
# ============================================================

print("\n[5/8] Starting BgUtils HTTP PO-token server...")

# Use the official Deno launch structure:
# cd node_modules
# deno run ... ../src/main.ts

log_file = open(SERVER_LOG, "w")

server_cmd = [
    deno,
    "run",
    "--allow-env",
    "--allow-net",
    "--allow-ffi=.",
    "--allow-read=.",
    "../src/main.ts",
    "--host",
    "127.0.0.1",
    "--port",
    str(BGUTIL_PORT),
]

print(">", " ".join(server_cmd))

server = subprocess.Popen(
    server_cmd,
    cwd=str(node_modules),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=deno_env,
)

# Wait up to 30 sec for port to open
server_ready = False

for i in range(60):
    if server.poll() is not None:
        break

    try:
        with socket.create_connection(
            ("127.0.0.1", BGUTIL_PORT),
            timeout=0.5
        ):
            server_ready = True
            break
    except:
        time.sleep(0.5)

if not server_ready:
    log_file.flush()

    print("\n========== BGUTIL SERVER LOG ==========")

    if SERVER_LOG.exists():
        print(SERVER_LOG.read_text(errors="ignore"))

    raise RuntimeError("BgUtils HTTP server failed to start")

print(
    f"BgUtils server READY at "
    f"http://127.0.0.1:{BGUTIL_PORT}"
)


# ============================================================
# 6. VERIFY PROVIDER
# ============================================================

print("\n[6/8] Provider setup complete.")

print("yt-dlp version:")
run([
    sys.executable,
    "-m",
    "yt_dlp",
    "--version",
])


# ============================================================
# 7. DOWNLOAD WITH AUTOMATIC FALLBACKS / RETRIES
# ============================================================

print("\n[7/8] Downloading audio...")

clean_audio()

# We know DEFAULT actually downloaded this exact video once before.
#
# Try several extraction strategies automatically instead of making
# you manually change commands every time.
#
# mweb + BgUtils is the current recommended POT configuration.
# web_embedded can work for embeddable videos without needing GVS POT.
# default lets yt-dlp choose its normal current clients.

strategies = [
    {
        "name": "mweb + BgUtils PO token",
        "extractor": "youtube:player_client=mweb",
    },
    {
        "name": "default clients + BgUtils",
        "extractor": "youtube:player_client=default",
    },
    {
        "name": "web_embedded + default",
        "extractor": "youtube:player_client=web_embedded,default",
    },
]

success = False
audio_path = None

# Two rounds gives us 6 real attempts.
# LOGIN_REQUIRED on Kaggle can be intermittent because the public
# datacenter egress IP can change/recover.
for round_num in range(1, 3):

    print("\n" + "#" * 70)
    print(f"DOWNLOAD ROUND {round_num}/2")
    print("#" * 70)

    for strategy in strategies:

        clean_audio()

        print("\n" + "=" * 70)
        print("TRYING:", strategy["name"])
        print("=" * 70)

        cmd = [
            sys.executable,
            "-m",
            "yt_dlp",

            "-v",
            "--no-playlist",
            "--force-ipv4",

            # JS challenge runtime
            "--js-runtimes",
            f"deno:{deno}",

            # BgUtils HTTP provider
            "--extractor-args",
            (
                "youtubepot-bgutilhttp:"
                f"base_url=http://127.0.0.1:{BGUTIL_PORT}"
            ),

            # YouTube player client for this attempt
            "--extractor-args",
            strategy["extractor"],

            # Retry media requests too
            "--retries", "5",
            "--fragment-retries", "5",
            "--retry-sleep", "2",

            # Audio only
            "-f",
            "bestaudio/best",

            "-o",
            OUTPUT_TEMPLATE,

            YOUTUBE_URL,
        ]

        start = time.time()

        result = subprocess.run(cmd)

        elapsed = time.time() - start

        audio_path = find_audio()

        if result.returncode == 0 and audio_path:
            print("\n✅ DOWNLOAD SUCCEEDED")
            print("Strategy:", strategy["name"])
            print("Time:", round(elapsed, 1), "seconds")
            success = True
            break

        print(
            f"\n❌ Attempt failed: {strategy['name']}"
        )

        # Small pause before a different YouTube client
        time.sleep(3)

    if success:
        break

    if round_num < 2:
        print("\nAll strategies failed once.")
        print("Waiting 10 seconds before retry round...")
        time.sleep(10)


# ============================================================
# 8. VERIFY ACTUAL AUDIO
# ============================================================

print("\n[8/8] Final verification...")

audio_path = find_audio()

if not success or not audio_path:

    print("\n" + "!" * 70)
    print("DOWNLOAD DID NOT SUCCEED")
    print("!" * 70)

    print(
        "\nThe BgUtils provider itself is installed and running."
    )

    print(
        "If every yt-dlp attempt above says LOGIN_REQUIRED / "
        "'Sign in to confirm you're not a bot', the remaining "
        "problem is the current Kaggle shared egress IP being "
        "blocked by YouTube — not a missing package."
    )

    print("\nBgUtils server log:")

    log_file.flush()

    if SERVER_LOG.exists():
        print(SERVER_LOG.read_text(errors="ignore")[-5000:])

else:

    size_mb = os.path.getsize(audio_path) / 1024 / 1024

    print("\n" + "=" * 70)
    print("✅ KAGGLE YOUTUBE AUDIO TEST PASSED")
    print("=" * 70)

    print("Audio file :", audio_path)
    print("Size       :", round(size_mb, 2), "MB")

    print("\nFFPROBE:")

    subprocess.run([
        "ffprobe",
        "-v", "error",
        "-show_entries",
        "format=duration,format_name,size",
        "-of",
        "default=noprint_wrappers=1",
        audio_path,
    ])

    print("\nYou can see/download the file from:")
    print(audio_path)

print("\nDONE.")

KAGGLE YOUTUBE AUDIO — FULL CLEAN BOOTSTRAP
Working directory: /kaggle/working

[1/8] Installing yt-dlp + BgUtils plugin...

> /usr/bin/python3 -m pip install -q -U yt-dlp[default]==2026.8.19 bgutil-ytdlp-pot-provider==2.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 7.4 MB/s eta 0:00:00

[2/8] Checking Deno...
Deno missing — installing Deno...

> bash -c curl -fsSL https://deno.land/install.sh | sh


######################################################################## 100.0%


Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Deno was installed successfully to /root/.deno/bin/deno
Run '/root/.deno/bin/deno --help' to get started

Stuck? Join our Discord https://discord.gg/deno
deno 2.9.6 (stable, release, x86_64-unknown-linux-gnu)

[3/8] Preparing BgUtils 2.0.0...

> git clone --depth 1 --single-branch --branch 2.0.0 https://github.com/Brainicism/bgutil-ytdlp-pot-provider.git /kaggle/working/bgutil-ytdlp-pot-provider


Cloning into '/kaggle/working/bgutil-ytdlp-pot-provider'...
Note: switching to '37169ee2656e08c5c2e5dc9df4c598c0cb4c88a8'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Download https://registry.npmjs.org/express/-/express-5.2.1.tgz
Download https://registry.npmjs.org/eslint/-/eslint-10.8.1.tgz
Download https://registry.npmjs.org/eslint-plugin-only-warn/-/eslint-plugin-only-warn-1.2.1.tgz
Download https://registry.npmjs.org/jsdom/-/jsdom-29.1.1.tgz
Download https://registry.npmjs.org/@typescript-eslint/parser/-/parser-8.67.


[4/8] Installing BgUtils Deno/npm dependencies...

> /root/.deno/bin/deno install --allow-scripts=npm:canvas --frozen


...

[youtube] Qd4jGgu06dw: Downloading player 8c3fda2d-main
[youtube] [jsc:deno] Solving JS challenges using deno


[debug] [youtube] Qd4jGgu06dw: Retrieved a subs PO Token for mweb client
[debug] Sort order given by extractor: quality, res, fps, hdr:12, source, vcodec, channels, acodec, lang, proto
[debug] Formats sorted by: hasvid, ie_pref, quality, res, fps, hdr:12(7), source, vcodec, channels, acodec, lang, proto, size, br, asr, vext, aext, hasaud, id


[info] Qd4jGgu06dw: Downloading 1 format(s): 251-1
[download] Sleeping 3.00 seconds as required by the site...


[debug] Invoking http downloader on "https://rr1---sn-qxo7rnee.googlevideo.com/videoplayback?expire=1789587114&ei=SpqqapKnDs6eybgP_92S-Ac&ip=34.71.100.231&id=o-AOEGYmO-LS5nK7UGjuoHg2Q5n58RAkcJqnXb4Dk2ZjUT&itag=251&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&cps=399&met=1789565514%2C&mh=pf&mm=18%2C29&mn=sn-qxo7rnee%2Csn-ojqxo5-5q&ms=aub%2Crdu&mv=u&mvi=1&pl=20&rms=aub%2Caub&pcm2=yes&bui=AR3QkAnW7R3G4avs12vzTOjOhD1lt6VFgxXaWuEFWtsQ74knUnBrF7T4oFrJv4g1zAw38jZehsm8giOR&spc=I-rgIQEukYu119XtP9vdx7XfTBBw42MvCx9ngx6lNbLIuKc2kfnLej3W1RmxN7tXZrDewsGrw0AUenrKvjD_sT5m2wU&vprv=1&svpuc=1&xtags=acont%3Doriginal%3Alang%3Dhi&mime=audio%2Fwebm&ns=z-VOekoZd0BoGKD3moU6n2AY&rqh=1&gir=yes&clen=9202591&dur=569.061&lmt=1766247036124589&mt=1789565106&fvip=4&keepalive=yes&fexp=51565115%2C52178456&c=MWEB&sefc=1&txp=4502534&n=YBAHigngZtlWnQ&sparams=expire%2Cei%2Cip%2Cid%2Citag%2Csource%2Crequiressl%2Cxpc%2Cpcm2%2Cbui%2Cspc%2Cvprv%2Csvpuc%2Cxtags%2Cmime%2Cns%2Crqh%2Cgir%2Cclen%2Cdur%2Clmt&sig=AE0s2JYwRQIhANb

[download] Destination: /kaggle/working/test_audio.webm
[download] 100% of    8.78MiB in 00:00:00 at 39.33MiB/s    

✅ DOWNLOAD SUCCEEDED
Strategy: mweb + BgUtils PO token
Time: 9.3 seconds

[8/8] Final verification...

✅ KAGGLE YOUTUBE AUDIO TEST PASSED
Audio file : /kaggle/working/test_audio.webm
Size       : 8.78 MB

FFPROBE:
format_name=matroska,webm
duration=569.061000
size=9202591

You can see/download the file from:
/kaggle/working/test_audio.webm

DONE.
